In [ ]:
# ---
# Visualize Top 20 GO Terms for Plasmodium malariae Genes
# Input: GO_terms.csv with Gene ↔ GO Term mappings
# Output: Barplot showing top 20 GO terms by frequency, saved as PNG
# ---


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
INPUT_FILE = "analysis_results/variants_analysis/files/GO_terms.csv"
OUTPUT_FIGURE = "analysis_results/variants_analysis/plots/top_20_GO_termss.png"

# Load & Prepare Data
df = pd.read_csv(INPUT_FILE)

# Define ontology colors
ontology_colors = {
    "Molecular Function": "#88CCEE",
    "Cellular Component": "#CC6677",
    "Biological Process": "#117733"
}

# Apply colors and count GO term occurrences
df["Color"] = df["Ontology"].map(ontology_colors)
go_counts = df["GO_Term"].value_counts()
df["GO_Count"] = df["GO_Term"].map(go_counts)

# Group by GO Term
df_grouped = df.groupby("GO_Term", as_index=False).agg({
    "GO_ID": lambda x: ", ".join(set(x)),
    "GO_Count": "first",
    "Ontology": "first"
})

# Select top 20 GO terms
df_grouped = df_grouped.sort_values(by="GO_Count", ascending=False).head(20)

# Plotting
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(
    data=df_grouped,
    y="GO_Term",
    x="GO_Count",
    hue="Ontology",
    palette=ontology_colors,
    ax=ax,
    edgecolor="black"
)

# Annotate with GO IDs
seen_go_ids = set()
for patch, (go_id, go_term) in zip(ax.patches, zip(df_grouped["GO_ID"], df_grouped["GO_Term"])):
    if go_id not in seen_go_ids:
        y_pos = patch.get_y() + patch.get_height() / 2
        x_pos = patch.get_x() + patch.get_width() + 0.1
        ax.text(x_pos, y_pos, go_id, fontsize=9, verticalalignment="center")
        seen_go_ids.add(go_id)

# Labels and formatting
ax.set_xlabel("GO Term Count", fontsize=12)
ax.set_ylabel("GO Terms", fontsize=12)
ax.set_title("Top 20 GO Terms ", fontsize=14)
ax.legend(title="Ontology", loc="lower right", fontsize=10)
plt.xscale("log")
plt.subplots_adjust(left=0.9, right=0.95, top=0.9, bottom=0.1)
plt.tight_layout()
plt.xticks([100, 200, 500, 1000], labels=["100", "200", "500", "1000"])

# Save & Show
plt.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved to {OUTPUT_FIGURE}")
